# Save the regenerated s1 dataset to Hugging Face

Run on **Colab CPU** with a **write-capable `HF_TOKEN`** in Colab Secrets.
This self-contained notebook updates `Seungjun/dp_removed_s1K-1.1` (`default`, `train`)
with `deepseek-v4-pro_reasoning` and `deepseek-v4-pro_answer`.
It reads your existing Drive export; it makes no DeepSeek requests.

All 996 rows and original fields are retained. The six unsuccessful rows keep null
Pro fields. No correctness, heading-format, or training-length filter is applied.
Both thinking and answer are saved separately for later combined training.
Existing `deepseek_grade` fields still describe the old answers, not the new Pro answers.

Run the cells in order: setup → paths → validate → upload → verify.
**The upload cell writes to the existing HF repository** with its existing visibility.
The dataset is published as Parquet, and a ZIP backs up the exact input JSONL files
plus a provenance manifest. A receipt with commit IDs is saved on Drive.
The original pinned revision remains the source for existing training notebooks;
this notebook does not change their dataset revision or column selections.

In [ ]:
# @title Install dependencies, mount Drive, and load your write token
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "PyYAML==6.0.3"])
from google.colab import drive, userdata
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Enable notebook access to a write-capable HF_TOKEN in Colab Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-s1-upload")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVzOu599cwQAAP8LAAAjAAAAcGlwZWxpbmUvcHVibGlzaF9zMV9yZWdlbmVyYXRpb24ucHmNVktv5DYMvs+vUFWgsNHpbLfHFHMqtr21C2wfh2AwUG3ao1aWvHrk0XT/+5KU7LGzmSS+ZCJK5MfvIylJKf9URrcqgvDQgwWPP1sR3gq4G52PQSjbigF8D8JZcy/iCYSFWzG4FoxonEmDDTsp5WbTeTcIdKUC4Dk90HmB7hMUW+OGwdmddpOx1T2EuNlsWuhykGMOWwWXfANb0STvwcZtgQPtVoSoYgoQ0Ohsp/v6aiPwQwjvPQTwNyDy6TflsOg0mDb8iBn+A00Ugw6Yc28HslF6LsUxIRp3a3uvWsj5kNMwQiP2JdC1zH7lgW0l94UZeTmW1bInHzh6d0v7jA5TZnVxkRACGsBWi63ZqLti/2rPOK4lcoD4oeVN8iCcLwyuDp/3I66IOR7zJnnIRNHnlQ6QtXnnvfOVfK+tJeHZEVIBQViHXKnYnFj0KfiksJxRKntfUQZz9Ay7Q3icuLaiekbHun4O14d1IUzRJ0dbFjC7EkMKUZwUFgB8TMpQ9IwlnMES0KIRY/2BWGT+Ic6GbHkO1rs71URqh1uHGoSoLRbWuX+m2lCe+upj0h7aM4ZlKPEN/5t53+XVo1UD8vJc/F++CMXJk2QeRqNQQ+d1r62am/Qc/xqJ2fUYtRT0UdsW7mQ9SUaKTeociAouXK9sDxXz+YJkWQ3WnlH9Dewxl5bzLXhxq+MJ2456MWjbv2nTaHRDYwix6AYmtDgyRgO4vBff8wJBZLhbUTmDZWT0DZYHtt5UUTUFA5sGJqiagf6nx2WfvDBbQvS6ifvffSq9uki5SIgxc7Phb4RSi/9Xyp73P0lTJ/+wc09hUoWJAuZcQZEVeeCcPxVWFo33L9yz6pgzMcHVjH+vcZ2lu6GItIpo14sreIVZ3LAtR9AhJrXTEYZQ1S9n89tUbtShohRSFrs5Uem0l3PhiDQhrwklVf+B4dAvAlLIOCwP6Bb3K2MqjbShbraBiv2wdjXPBf5/R1KOVa7uObccckVnFr80Bv/GltjvhZyKUK5JwCPEfMaCvpfnO211OB09qOCs5DJBn2585OIClz+VgO2SL5xsYa4SDBd9sg2PgHx/Lfh8IiG8GkdnAxz54pZlTud7Ky8dXgVtUbT5CXBRVPrm9v12L97OFjCleIsa+ar51Vl4UqNXNFJIDc6M0CWzYux8IdhkzGI+5xfBAuwT1bYO+7X4oDoQ0eF8ReIRvtAtTg8cWkbg8HKKa6DTGGf0cKNdCng/THoh/cnEsHssEfUqC0TB6xUVVMDcyrkfUK+5OV6l1F8umXa+DRAv3PFF1S94yFSvJbwSDxRkkpKfZdRrZVzOU/giXdQXxZYPr661NfbZe9npYXA3ML2iMiubS5tV25advHHLF9tifJSrbBrvh63oADvCw56Joi73yIescwgPaJswb8WD5GfWVX5CbJdjYK5rXMWbf7lOL5/vzvYvhKJv9VC8mghEX+VxMz3YrqbnHd+/GVddX/DZOJSniRbb4IhVGMHj+Z+VCQQSHzh9PD1a/7T5DFBLAwQUAAAACAAAACFcU3WED2MBAADbAgAAJAAAAGNvbmZpZ3MvcHVibGlzaF9zMV9yZWdlbmVyYXRpb24ueWFtbLWRy27cIBSG934Kv4AHg20cvKuUKorarOYBrGM4uKQMIC4zzduXmYm6iCpVWXQFnMv/nf+QfIkSl6ZtIwa/tEcsbn8tjqiwRjz5M6o10W8dPdBbzdkk493SckXnedO8ByomJQfJtnFQyOEBtaZs7CegUoysNknvtNmXVqGGYnONpGBNXtocwbj6xF8BZa6g6C9paYXg966MLq/K7JhqMZ+nkekJ9cC0wFkP89TDsImKwmns+axmlHPfj5QzwAc1gGQ9E1wwAZoPU5Mh7pg/7/R0H/IfLk5eob1mMSTEn9157EL0jcPLKr0tJ5eu5O5jQSVD8s64/a9ZcOmCsdHG4q1fQYZUTbTkfT1ERXNG8vL2eDu/P3Vfnl++dkfaPVapY5UisbhEPgiTSY9U1hVRuWm5cUV0sZa8yx8CxGzAHl7raPbqNEMu6X9R7+p/YBElmnD9+HrLPr59ivtDryVYDyo1vwFQSwECFAMUAAAACAAAACFcAAAAAAIAAAAAAAAAEgAAAAAAAAAAAAAAgAEAAAAAY29tbW9uL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAAAAAAAAAAAAAIABMgAAAGNvbW1vbi9pby5weVBLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAAUAAAAAAAAAAAAAACAATgDAABwaXBlbGluZS9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVzOu599cwQAAP8LAAAjAAAAAAAAAAAAAACAAWwDAABwaXBlbGluZS9wdWJsaXNoX3MxX3JlZ2VuZXJhdGlvbi5weVBLAQIUAxQAAAAIAAAAIVxTdYQPYwEAANsCAAAkAAAAAAAAAAAAAACAASAIAABjb25maWdzL3B1Ymxpc2hfczFfcmVnZW5lcmF0aW9uLnlhbWxQSwUGAAAAAAUABQBfAQAAxQkAAAAA'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

In [ ]:
# @title Settings — defaults point to your existing full-run export
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/publish_s1_regeneration.yaml").read_text())
# If you move the files or finish the remaining six rows, change these paths here.
# CONFIG["files"]["dataset"] = "/content/drive/MyDrive/.../dataset.jsonl"
# CONFIG["files"]["status"] = "/content/drive/MyDrive/.../status.jsonl"
print(yaml.safe_dump(CONFIG, sort_keys=False))

In [ ]:
# @title Read files, verify row alignment, and prepare the two added columns
import hashlib, json, shutil
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from common.io import digest, read_jsonl, write_json
from pipeline.publish_s1_regeneration import merge_export

api = HfApi(token=HF_TOKEN)
spec, target = CONFIG["source"], CONFIG["target"]
# Capture immutable inputs before validation and upload.
snapshot = Path("/content/s1-upload-snapshot")
snapshot.mkdir(parents=True, exist_ok=True)
for key, filename in [("dataset", "dataset.partial.jsonl"), ("status", "status.jsonl")]:
    path = Path(CONFIG["files"][key])
    if not path.is_file():
        raise FileNotFoundError(f"Update CONFIG['files']['{key}']; missing: {path}")
    shutil.copyfile(path, snapshot / filename)
exported = read_jsonl(snapshot / "dataset.partial.jsonl")
statuses = read_jsonl(snapshot / "status.jsonl")
source = load_dataset(spec["repo"], name=spec["config"], split=spec["split"],
                      revision=spec["revision"], token=HF_TOKEN)
before = api.dataset_info(target["repo"], revision=target["revision"]).sha
current = load_dataset(target["repo"], name=target["config"], split=target["split"],
                       revision=before, token=HF_TOKEN)
merged, summary = merge_export(source, current, exported, statuses, CONFIG)
manifest = {"created_at": datetime.now(timezone.utc).isoformat(),
            "source": spec, "target": target, "previous_revision": before,
            "model": CONFIG["model"], "summary": summary,
            "original_paths": CONFIG["files"],
            "sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in snapshot.iterdir() if p.suffix == ".jsonl"}}
write_json(snapshot / "manifest.json", manifest)
archive_path = Path("/content/s1-generation-backup.zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for filename in ["dataset.partial.jsonl", "status.jsonl", "manifest.json"]:
        archive.write(snapshot / filename, arcname=filename)
archive_sha = hashlib.sha256(archive_path.read_bytes()).hexdigest()
backup_remote = f"generation_backups/{CONFIG['model']}/{archive_sha}.zip"
receipt_dir = Path(CONFIG["files"]["receipt_directory"])
receipt_dir.mkdir(parents=True, exist_ok=True)
receipt_path = receipt_dir / f"{archive_sha}.json"
receipt = {**manifest, "backup_path": backup_remote, "backup_sha256": archive_sha,
           "verified": False}
write_json(receipt_path, receipt)
print(json.dumps(summary, indent=2))
print("Target:", target["repo"], target["config"], target["split"])
print("Existing revision:", before)
print("Backup:", backup_remote)

## Upload the updated dataset and backup

Run this cell to publish the prepared result. Identical reruns are supported;
validation also permits filling missing Pro results later, but refuses to erase
or replace existing non-null Pro text with different text.
If a connection fails, rerun validation and upload. Use one uploader at a time.
The receipt records each completed upload before verification.

In [ ]:
# @title Push the two added columns and back up both input files
if api.dataset_info(target["repo"], revision=target["revision"]).sha != before:
    raise RuntimeError("HF changed since validation. Rerun the validation cell first.")
commit = merged.push_to_hub(
    target["repo"], config_name=target["config"], split=target["split"],
    revision=target["revision"], token=HF_TOKEN,
    commit_message="Add DeepSeek Pro reasoning and answers; preserve all source rows")
receipt["dataset_revision"] = commit.oid
write_json(receipt_path, receipt)
backup_commit = api.upload_file(
    path_or_fileobj=str(archive_path), path_in_repo=backup_remote,
    repo_id=target["repo"], repo_type="dataset", revision=target["revision"],
    commit_message="Back up regeneration JSONL, row status, and provenance")
receipt["revision"] = backup_commit.oid
write_json(receipt_path, receipt)
print("Uploaded:", f"https://huggingface.co/datasets/{target['repo']}/tree/{backup_commit.oid}")
print("Receipt saved:", receipt_path)

In [ ]:
# @title Reload the uploaded revision and verify all rows plus the backup
receipt = json.loads(receipt_path.read_text())
revision = receipt["revision"]
uploaded = load_dataset(target["repo"], name=target["config"], split=target["split"],
                        revision=revision, token=HF_TOKEN)
if digest(list(uploaded)) != summary["dataset_digest"]:
    raise ValueError("Uploaded dataset does not exactly match the prepared dataset")
downloaded = hf_hub_download(repo_id=target["repo"], filename=backup_remote,
                             repo_type="dataset", revision=revision, token=HF_TOKEN)
if hashlib.sha256(Path(downloaded).read_bytes()).hexdigest() != archive_sha:
    raise ValueError("Uploaded backup checksum mismatch")
receipt["verified"] = True
write_json(receipt_path, receipt)
print("Verified:", len(uploaded), "rows; original fields, both Pro columns, and backup intact.")
print("Pinned revision for later use:", revision)
print("Drive receipt:", receipt_path)
print(f"https://huggingface.co/datasets/{target['repo']}")

The six null pairs are intentionally retained for backup, not silently treated as training
examples. Publishing does not judge correctness or shorten either output column.
No API keys are included in the backup or receipt.

References: [Dataset.push_to_hub](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.push_to_hub),
[Hub file uploads](https://huggingface.co/docs/huggingface_hub/guides/upload).